# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulah-naeem/FlyRank-ml-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

This playbook takes pages flagged as decaying by our ML model and routes them to four primary actions:
1. **Deep Refresh:** Content is decaying but has high historical traffic (`impressions_prev_30d` > threshold).
2. **Light Update:** Minor decay, low effort to fix (e.g., metadata tweak).
3. **Consolidate:** Low traffic, cannibalizing other pages.
4. **Archive:** Zero value over 90 days.

The reason codes explain *why* the model flagged it, giving the human reviewer immediate context.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Simulate the scored output from W06 for the playbook queue
np.random.seed(42)
n_samples = 200
action_queue = pd.DataFrame({
    'content_id': [f'content_{i}' for i in range(n_samples)],
    'client_id': np.random.choice(['client_A', 'client_B', 'client_C'], n_samples),
    'model_score': np.random.uniform(0.6, 0.99, n_samples),
    'impressions_prev_30d': np.random.exponential(1000, n_samples)
})

# Map archetypes to actions based on thresholds
def assign_action(row):
    if row['impressions_prev_30d'] > 1500:
        return 'Deep Refresh', 'High historical volume, severe decay'
    elif row['impressions_prev_30d'] > 500:
        return 'Light Update', 'Moderate volume, slipping ranks'
    elif row['impressions_prev_30d'] > 100:
        return 'Consolidate', 'Low volume, cannibalization risk'
    else:
        return 'Archive', 'Zero value, dead weight'

action_queue[['recommended_action', 'reason_code']] = action_queue.apply(assign_action, axis=1, result_type='expand')
action_queue = action_queue.sort_values('model_score', ascending=False).reset_index(drop=True)

print("Top 5 Prioritized Actions for Human Review:")
display(action_queue.head(5))


## 2. Intended use and limits

**Intended Use:** This playbook is a directional decision-support tool for the editorial team. It prioritizes which 50 pages they should review first out of thousands, saving hours of manual analytics digging.

**Limits:** 
- It does not make automated edits.
- It relies on trailing 90-day data, meaning it cannot detect viral spikes or algorithm penalties that happened yesterday.
- It is not causal; it observed patterns of decay but does not prove *why* the decay happened.

In [ ]:
# No code needed for this section.

## 3. Human review + the no-go list

**Human Review Rules:** An editor must manually verify the page intent hasn't fundamentally shifted before applying a Deep Refresh.

**No-Go List (Never Automate):**
1. Deleting or archiving pages (always requires human sign-off to prevent catastrophic traffic loss).
2. Modifying or redirecting core legal/compliance pages, regardless of their decay score.
3. Publishing AI-rewritten content without a human-in-the-loop editorial pass.

In [ ]:
# No code needed for this section.

## 4. Monitoring / retrain triggers

We must monitor the model to ensure recommendations do not go stale.

**Retrain Triggers:**
1. **Performance Drop:** If Precision@50 on the holdout validation set drops below the 44% baseline for two consecutive weeks.
2. **Data Drift:** If a major Google core update fundamentally shifts the underlying distributions (e.g., `avg_position` or `ai_traffic_pct` drifts significantly beyond the training distribution).

In [ ]:
# No code needed for this section.

## 5. Exports for the paper

Exporting the ranked queue CSV to `work/outputs/` (which is excluded from Git to prevent data leakage) and generating figures for the research paper in `work/figures/`.

In [ ]:
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# 1. Export the queue (Data stays out of Git)
action_queue.to_csv('../outputs/action_queue.csv', index=False)
print("Exported action_queue.csv to work/outputs/")

# 2. Export a figure for the research paper (Committed to Git)
plt.figure(figsize=(8, 5))
action_queue['recommended_action'].value_counts().plot(kind='bar', color='#1E293B')
plt.title('Distribution of Recommended Content Actions', pad=15)
plt.ylabel('Number of Pages')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../figures/action_distribution.png', dpi=300)
print("Exported action_distribution.png to work/figures/")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.